# Stage 1B (Colab): Tile Province Images to Prediction Points

Runs on Google Colab with Drive mounted. Processes one province at a time with checkpointing so session timeouts are recoverable.

**Drive structure expected:**
```
My Drive/Thesis/
  2025-Data/
    prediction_points.csv
    Sentinel2/
      Ilocos_Norte/
        Ilocos_Norte_2025_Q1.tif
        ...
      Pampanga/
        ...
    tiles/              <- output goes here
    tile_checkpoint.csv <- tracks completed points
```

In [ ]:
# ============================================================
# 0. MOUNT DRIVE AND INSTALL RASTERIO
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install rasterio pyproj -q

import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds
from pyproj import Transformer
import time
import warnings
warnings.filterwarnings('ignore')

print("Ready.")

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

DRIVE_ROOT = '/content/drive/MyDrive/Thesis/2025-Data'
POINTS_CSV = f'{DRIVE_ROOT}/prediction_points.csv'

# All images are directly in this folder now (no subfolders)
IMAGE_ROOT = f'{DRIVE_ROOT}/Sentinel2'

# Save tiles locally to Colab first for lightning-fast processing!
TILE_OUTPUT = '/content/tiles' 
# Save the final zipped files to Drive to prevent Google API rate limits
DRIVE_OUTPUT_DIR = f'{DRIVE_ROOT}/tiles_zipped'
CHECKPOINT  = f'{DRIVE_ROOT}/tile_checkpoint.csv'

os.makedirs(TILE_OUTPUT, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

TILE_HALF_SIZE_M = 5000  # 10km x 10km tiles
QUARTERS = ['Q1', 'Q2', 'Q3', 'Q4']

# Maps Province Name (from your CSV) -> Exact File Prefix
PROVINCE_PREFIX_MAP = {
    'Ilocos Norte':         'Ilocos_Norte',
    'Pampanga':             'Pampanga',
    'Benguet':              'Benguet',
    'Kalinga':              'Kalinga',
    'Aklan':                'Aklan',
    'Zamboanga del Norte':  'Zamboanga_del_Norte',
    'Basilan':              'Basilan',
    'Tawi-Tawi':            'Tawi-Tawi',
    'Maguindanao del Sur':  'Maguindanao_del_Sur',
    'Davao Oriental':       'Davao_Oriental',
    'NCR':                  'NCR'  
}

print(f"Configured {len(PROVINCE_PREFIX_MAP)} provinces in a flat directory.")

In [ ]:
# ============================================================
# 2. VERIFY IMAGE FILES ON DRIVE
# ============================================================

print("Checking image files on Drive...")
ok_count = 0
missing = []

for prov, prefix in PROVINCE_PREFIX_MAP.items():
    for q in QUARTERS:
        # Construct the flat filename: e.g., "Ilocos_Norte_2025_Q1.tif"
        fname = f"{prefix}_2025_{q}.tif"
        fpath = os.path.join(IMAGE_ROOT, fname)
        
        if os.path.exists(fpath):
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"  OK  : {fname} ({size_mb:.0f} MB)")
            ok_count += 1
        else:
            print(f"  MISS: {fname}")
            missing.append((prov, q))

print(f"\nFound: {ok_count}, Missing: {len(missing)}")
if missing:
    print("Fix missing files before proceeding.")

In [ ]:
# ============================================================
# 3. LOAD POINTS AND CHECKPOINT
# ============================================================

df_points = pd.read_csv(POINTS_CSV)
print(f"Total prediction points: {len(df_points)}")
print(f"Points per province:")
print(df_points['Province'].value_counts().to_string())

# Load checkpoint of already-completed points
if os.path.exists(CHECKPOINT):
    df_done = pd.read_csv(CHECKPOINT)
    done_ids = set(df_done['PointID'].tolist())
    print(f"\nCheckpoint found: {len(done_ids)} points already tiled.")
else:
    done_ids = set()
    print("\nNo checkpoint found. Starting fresh.")

remaining = df_points[~df_points['PointID'].isin(done_ids)]
print(f"Remaining: {len(remaining)} points")

In [ ]:
# ============================================================
# 4. TILING FUNCTION
# ============================================================

to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:32651', always_xy=True)

def crop_tile(src_path, center_lon, center_lat, half_size_m, out_path):
    """
    Crop a tile from a province GeoTIFF centered on (lon, lat).
    """
    cx, cy = to_utm.transform(center_lon, center_lat)
    minx_utm = cx - half_size_m
    maxx_utm = cx + half_size_m
    miny_utm = cy - half_size_m
    maxy_utm = cy + half_size_m

    with rasterio.open(src_path) as src:
        src_crs = src.crs

        # ENFORCE STRICT UTM EXPECTATION to prevent CNN distortion
        if str(src_crs) != 'EPSG:32651':
            raise ValueError(f"CRITICAL ERROR: Image {src_path} is not in EPSG:32651. Native CRS is {src_crs}.")
            
        minx_s, miny_s = minx_utm, miny_utm
        maxx_s, maxy_s = maxx_utm, maxy_utm

        # Clamp to source bounds
        b = src.bounds
        minx_s = max(minx_s, b.left)
        miny_s = max(miny_s, b.bottom)
        maxx_s = min(maxx_s, b.right)
        maxy_s = min(maxy_s, b.top)

        if minx_s >= maxx_s or miny_s >= maxy_s:
            return False

        window = from_bounds(
            minx_s, miny_s, maxx_s, maxy_s,
            transform=src.transform
        )

        n_bands = min(src.count, 3)
        data = src.read(
            list(range(1, n_bands + 1)),
            window=window
        )

        if data.size == 0 or data.shape[1] == 0 or data.shape[2] == 0:
            return False

        out_transform = rasterio.transform.from_bounds(
            minx_s, miny_s, maxx_s, maxy_s,
            data.shape[2], data.shape[1]
        )

        profile = src.profile.copy()
        profile.update({
            'width': data.shape[2],
            'height': data.shape[1],
            'count': n_bands,
            'transform': out_transform,
            'compress': 'lzw',
        })

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data)

    return True

print("Tiling function ready. Strict UTM enforced.")

In [ ]:
# ============================================================
# 5. TILE ONE PROVINCE AT A TIME
# ============================================================
# This is the main processing cell. It processes one province
# per run. After each province, results are checkpointed to
# Drive. If the session dies, re-run from this cell.
#
# To process a specific province, set CURRENT_PROVINCE.
# To auto-pick the next unfinished province, set to None.

CURRENT_PROVINCE = None  # Set to e.g. 'Ilocos Norte' or None for auto

# ---- Auto-select next province ----
if CURRENT_PROVINCE is None:
    remaining_provs = remaining['Province'].unique()
    if len(remaining_provs) == 0:
        print("All provinces complete!")
    else:
        CURRENT_PROVINCE = remaining_provs[0]
        print(f"Auto-selected: {CURRENT_PROVINCE}")

if CURRENT_PROVINCE and CURRENT_PROVINCE in PROVINCE_IMAGE_MAP:
    info = PROVINCE_IMAGE_MAP[CURRENT_PROVINCE]
    prov_points = remaining[remaining['Province'] == CURRENT_PROVINCE]
    n_points = len(prov_points)

    print(f"\nProcessing: {CURRENT_PROVINCE}")
    print(f"Points to tile: {n_points}")
    print(f"Total tiles: {n_points * 4}")

    # Open all 4 quarterly source images once (big speedup)
    src_paths = {}
    for q in QUARTERS:
        src_paths[q] = os.path.join(
            IMAGE_ROOT, info['folder'],
            info['pattern'].format(q=q)
        )

    completed = []
    failed_list = []
    t_start = time.time()

    for i, (_, row) in enumerate(prov_points.iterrows()):
        pid = int(row['PointID'])
        lat = row['Latitude']
        lon = row['Longitude']
        pid_str = str(pid).zfill(5)

        out_dir = os.path.join(TILE_OUTPUT, pid_str)
        os.makedirs(out_dir, exist_ok=True)

        all_ok = True
        for q in QUARTERS:
            out_path = os.path.join(
                out_dir, f'point_{pid_str}_2025_{q}.tif'
            )

            if os.path.exists(out_path):
                continue

            try:
                ok = crop_tile(
                    src_paths[q], lon, lat,
                    TILE_HALF_SIZE_M, out_path
                )
                if not ok:
                    all_ok = False
                    failed_list.append({
                        'PointID': pid, 'Quarter': q,
                        'Error': 'no_overlap'
                    })
            except Exception as e:
                all_ok = False
                failed_list.append({
                    'PointID': pid, 'Quarter': q,
                    'Error': str(e)[:80]
                })

        completed.append({'PointID': pid, 'Province': CURRENT_PROVINCE})

        # Progress every 25 points
        if (i + 1) % 25 == 0 or (i + 1) == n_points:
            elapsed = time.time() - t_start
            rate = (i + 1) / elapsed * 60
            eta = (n_points - i - 1) / (rate / 60) if rate > 0 else 0
            print(f"  {i+1}/{n_points} points "
                  f"({rate:.0f} pts/min, ETA {eta:.0f}s)")

    # ---- Save checkpoint ----
    df_new = pd.DataFrame(completed)
    if os.path.exists(CHECKPOINT):
        df_prev = pd.read_csv(CHECKPOINT)
        df_ckpt = pd.concat([df_prev, df_new], ignore_index=True)
    else:
        df_ckpt = df_new

    df_ckpt.to_csv(CHECKPOINT, index=False)

    elapsed_total = time.time() - t_start
    print(f"\n{CURRENT_PROVINCE} complete.")
    print(f"  Points: {len(completed)}")
    print(f"  Failed tiles: {len(failed_list)}")
    print(f"  Time: {elapsed_total:.0f}s")
    print(f"  Checkpoint saved: {CHECKPOINT}")

    if failed_list:
        df_fail = pd.DataFrame(failed_list)
        fail_path = f'{TILE_OUTPUT}/failed_{CURRENT_PROVINCE.replace(" ", "_")}.csv'
        df_fail.to_csv(fail_path, index=False)
        print(f"  Failures log: {fail_path}")

elif CURRENT_PROVINCE:
    print(f"Province '{CURRENT_PROVINCE}' not in image map. Check spelling.")

In [ ]:
# ============================================================
# 5B. LOOP: AUTO-PROCESS ALL REMAINING PROVINCES
# ============================================================
import shutil

if os.path.exists(CHECKPOINT):
    df_done_reload = pd.read_csv(CHECKPOINT)
    done_ids_reload = set(df_done_reload['PointID'].tolist())
else:
    done_ids_reload = set()

remaining_reload = df_points[~df_points['PointID'].isin(done_ids_reload)]
remaining_provs = remaining_reload['Province'].unique()
print(f"Provinces remaining: {len(remaining_provs)}")

for prov_name in remaining_provs:
    if prov_name not in PROVINCE_PREFIX_MAP:
        print(f"\nSKIP: '{prov_name}' not in image map.")
        continue

    prefix = PROVINCE_PREFIX_MAP[prov_name]
    prov_pts = remaining_reload[remaining_reload['Province'] == prov_name]

    print(f"\n{'='*50}")
    print(f"Processing: {prov_name} ({len(prov_pts)} points)")
    print(f"{'='*50}")

    src_paths = {}
    for q in QUARTERS:
        fname = f"{prefix}_2025_{q}.tif"
        src_paths[q] = os.path.join(IMAGE_ROOT, fname)

    completed = []
    failed_list = []
    t_start = time.time()

    for i, (_, row) in enumerate(prov_pts.iterrows()):
        pid = int(row['PointID'])
        pid_str = str(pid).zfill(5)
        out_dir = os.path.join(TILE_OUTPUT, pid_str)
        os.makedirs(out_dir, exist_ok=True)

        for q in QUARTERS:
            out_path = os.path.join(out_dir, f'point_{pid_str}_2025_{q}.tif')
            if os.path.exists(out_path):
                continue
            try:
                ok = crop_tile(
                    src_paths[q], row['Longitude'],
                    row['Latitude'], TILE_HALF_SIZE_M, out_path
                )
                if not ok:
                    failed_list.append({'PointID': pid, 'Quarter': q, 'Error': 'no_overlap'})
            except Exception as e:
                failed_list.append({'PointID': pid, 'Quarter': q, 'Error': str(e)[:80]})

        completed.append({'PointID': pid, 'Province': prov_name})

        if (i + 1) % 50 == 0 or (i + 1) == len(prov_pts):
            elapsed = time.time() - t_start
            rate = (i + 1) / elapsed * 60 if elapsed > 0 else 0
            print(f"  {i+1}/{len(prov_pts)} ({rate:.0f} pts/min)")

    # Zip the local folder and move it to Drive to prevent Google API rate limits!
    print(f"  Packaging tiles for {prov_name}...")
    zip_name = f"tiles_{prefix}"
    zip_path_local = f"/content/{zip_name}"
    shutil.make_archive(zip_path_local, 'zip', TILE_OUTPUT)
    
    # Move the ZIP to Drive
    drive_zip_target = os.path.join(DRIVE_OUTPUT_DIR, f"{zip_name}.zip")
    shutil.move(f"{zip_path_local}.zip", drive_zip_target)
    print(f"  ✓ Zipped and saved to Drive: {drive_zip_target}")
    
    # Clear the local Colab folder for the next province
    shutil.rmtree(TILE_OUTPUT)
    os.makedirs(TILE_OUTPUT, exist_ok=True)

    # Checkpoint after each province
    df_new = pd.DataFrame(completed)
    if os.path.exists(CHECKPOINT):
        df_prev = pd.read_csv(CHECKPOINT)
        df_ckpt = pd.concat([df_prev, df_new], ignore_index=True)
    else:
        df_ckpt = df_new
    df_ckpt.to_csv(CHECKPOINT, index=False)

    elapsed_total = time.time() - t_start
    print(f"  Done: {len(completed)} pts, {len(failed_list)} failed tiles, {elapsed_total:.0f}s")

print(f"\nAll provinces processed safely.")

In [ ]:
# ============================================================
# 6. VERIFY OUTPUT ON DRIVE
# ============================================================

tile_counts = []
for pid_str in os.listdir(TILE_OUTPUT):
    d = os.path.join(TILE_OUTPUT, pid_str)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.endswith('.tif')])
        tile_counts.append({'PointID': pid_str, 'n_tiles': n})

df_counts = pd.DataFrame(tile_counts)

print(f"{'='*50}")
print(f"TILING VERIFICATION")
print(f"{'='*50}")
print(f"Total point folders: {len(df_counts)}")
print(f"Complete (4/4):      {(df_counts['n_tiles'] == 4).sum()}")
print(f"Incomplete (<4):     {(df_counts['n_tiles'] < 4).sum()}")
print(f"Empty (0):           {(df_counts['n_tiles'] == 0).sum()}")
print(f"Total tiles on disk: {df_counts['n_tiles'].sum()}")

if (df_counts['n_tiles'] < 4).any():
    print("\nIncomplete points:")
    print(df_counts[df_counts['n_tiles'] < 4].to_string(index=False))

In [ ]:
# ============================================================
# 7. VISUAL SPOT CHECK
# ============================================================

import matplotlib.pyplot as plt

# Pick 3 random complete points
complete_dirs = df_counts[df_counts['n_tiles'] == 4]['PointID'].tolist()
samples = np.random.choice(complete_dirs, min(3, len(complete_dirs)), replace=False)

fig, axes = plt.subplots(len(samples), 4, figsize=(16, 4 * len(samples)))
if len(samples) == 1:
    axes = axes.reshape(1, -1)

for row_i, pid_str in enumerate(samples):
    d = os.path.join(TILE_OUTPUT, pid_str)
    tifs = sorted([f for f in os.listdir(d) if f.endswith('.tif')])

    for col_i, tif in enumerate(tifs[:4]):
        ax = axes[row_i, col_i]
        with rasterio.open(os.path.join(d, tif)) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0)).astype(float)
            p2, p98 = np.percentile(img, (2, 98))
            if p98 > p2:
                img = np.clip((img - p2) / (p98 - p2), 0, 1)
            ax.imshow(img)
            ax.set_title(
                tif.split('_')[-1].replace('.tif', ''),
                fontsize=10
            )
            ax.axis('off')
    axes[row_i, 0].set_ylabel(
        f'Point {pid_str}', fontsize=11, rotation=0,
        labelpad=60, va='center'
    )

plt.suptitle('Tile spot check (random samples)', fontsize=13)
plt.tight_layout()
plt.show()

## Next: VGG16 Feature Extraction

The tiles are now on Drive. You can run VGG16 feature extraction in the same Colab session (or a new one with GPU) by pointing it at the `tiles/` folder on Drive. The tiles stay on Drive throughout, nothing needs to download to your Mac.